## 7. Backtesting Quantitativo e Valutazione Finanziaria

Nel trading algoritmico, le metriche di classificazione standard del Machine Learning (Accuracy, Precision) risultano insufficienti per validare un modello. Un'elevata accuratezza direzionale può tradursi in perdite economiche repentine se i profitti medi per operazione sono inferiori ai costi di transazione imposti dal mercato.

Questo modulo implementa un motore di backtesting vettorializzato Out-of-Sample. Il sistema converte i segnali direzionali del modello XGBoost in rendimenti monetari reali, applicando logiche rigorose di capitalizzazione, dimensionamento della posizione (Lottaggio) e calcolando l'impatto dei costi operativi (Spread) sull'intera serie storica di test.

### 7.1 Allineamento Dati e Addestramento del Modello Predittivo
La pipeline richiede un allineamento perfetto tra le feature geometriche (Matrice $X$) e i prezzi reali di mercato (DataFrame). I dati vengono nuovamente suddivisi tramite split cronologico (80/20) per garantire che la simulazione di portafoglio avvenga esclusivamente su dati invisibili al modello durante la fase di fitting.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

"""
Obiettivo: Preparare l'ambiente di simulazione finanziaria (Backtest) e generare i segnali operativi.
In breve: Il modulo riutilizza la logica delle Sliding Windows per allineare temporalmente 
i prezzi reali alle feature estratte. Successivamente, addestra il classificatore XGBoost e mappa 
le predizioni (Rialzista/Ribassista) esclusivamente sul segmento futuro (Test Set), 
creando la base dati per la simulazione del portafoglio.
"""

def prepare_backtest_data(df, window_size=10):
    """
    Rielabora la logica delle sliding windows restituendo, oltre a X e y, 
    un DataFrame perfettamente allineato per il calcolo del Profit & Loss finanziario.
    """
    df_features = df.copy()
    
    # Feature spaziali stazionarie
    df_features['Body'] = df_features['Close'] - df_features['Open']
    df_features['Range'] = df_features['High'] - df_features['Low']
    df_features['Upper_Shadow'] = df_features['High'] - df_features[['Open', 'Close']].max(axis=1)
    df_features['Lower_Shadow'] = df_features[['Open', 'Close']].min(axis=1) - df_features['Low']
    
    # Label: 0 = Rialzista, 1 = Ribassista
    df_features['Target'] = np.where(df_features['Body'] >= 0, 0, 1)
    
    X, y = [], []
    features_array = df_features[['Body', 'Range', 'Upper_Shadow', 'Lower_Shadow']].values
    target_array = df_features['Target'].values
    
    for i in range(len(df_features) - window_size):
        X.append(features_array[i : i + window_size].flatten())
        y.append(target_array[i + window_size])
        
    # Allineamento del DataFrame dei prezzi al vettore temporale ridotto
    df_aligned = df_features.iloc[window_size:].copy().reset_index(drop=True)
        
    return np.array(X), np.array(y), df_aligned

# Configurazione path per esecuzione in ambiente Notebook (Root directory)
file_path = os.path.join(os.getcwd(), 'Data Management', 'ReadyData', 'XAUUSD_ReadyToUse.csv')
print("[SYSTEM] Caricamento storico e allineamento per backtest in corso...")

df = pd.read_csv(file_path, parse_dates=['Datetime'])
df_clean = df[df['Missing'] == False].copy().reset_index(drop=True)

X, y, df_aligned = prepare_backtest_data(df_clean, window_size=10)

# Split Cronologico (80% Train, 20% Test)
split_index = int(len(X) * 0.8)

X_train, y_train = X[:split_index], y[:split_index]
X_test, y_test = X[split_index:], y[split_index:]

# Isolamento del segmento Out-of-Sample per la simulazione del capitale
df_test = df_aligned.iloc[split_index:].copy().reset_index(drop=True)

print("[SYSTEM] Addestramento XGBoost Classifier su porzione Training (80%)...")
xgb_model = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

print("[SYSTEM] Popolamento segnali direzionali su porzione Testing (20%)...")
df_test['Prediction'] = xgb_model.predict(X_test)

### 7.2 Motore di Simulazione Vettorializzato
L'elaborazione delle performance di portafoglio non utilizza cicli iterativi lenti, ma applica operazioni matriciali (vettorizzazione) direttamente sull'intero Test Set (dati Out-of-Sample). I parametri finanziari fondamentali impostati sono:
*   **Capitale Iniziale:** 10.000 Dollari.
*   **Lottaggio:** 0.1 Lotti (equivalenti a un controvalore di $10 per ogni punto pieno di variazione del prezzo).
*   **Spread:** 0.15 Punti base. Tale valore viene detratto sistematicamente dal rendimento lordo di ogni singola transazione.

L'equazione base per il calcolo del rendimento netto in punti è definita come $PnL_{net} = (Movimento \times Posizione) - Spread$.

In [ ]:
print("[SYSTEM] Avvio simulazione di portafoglio vettorializzata...")

# Parametri Finanziari
INITIAL_CAPITAL = 10000.0   
LOT_SIZE = 0.1              
CONTRACT_MULTIPLIER = 100 * LOT_SIZE 
SPREAD = 0.15               

# Decodifica previsioni in posizionamento di mercato (+1 Long, -1 Short)
df_test['Position'] = np.where(df_test['Prediction'] == 0, 1, -1)

# Calcolo del Movimento Lordo (Delta tra chiusura e apertura della candela)
df_test['Market_Move'] = df_test['Close'] - df_test['Open']

# Calcolo Profit & Loss in Punti (Lordo)
df_test['Gross_PnL_Points'] = df_test['Market_Move'] * df_test['Position']

# Calcolo Profit & Loss in Punti (Netto, post-spread)
df_test['Net_PnL_Points'] = df_test['Gross_PnL_Points'] - SPREAD

# Conversione Punti in Controvalore Monetario (USD)
df_test['Net_PnL_USD'] = df_test['Net_PnL_Points'] * CONTRACT_MULTIPLIER

# Vettorizzazione della Curva di Equity Cumulativa
df_test['Equity_Curve'] = INITIAL_CAPITAL + df_test['Net_PnL_USD'].cumsum()

# Estrazione metriche di Rischio (Peak e Max Drawdown)
df_test['Peak'] = df_test['Equity_Curve'].cummax()
df_test['Drawdown'] = df_test['Equity_Curve'] - df_test['Peak']
max_drawdown = df_test['Drawdown'].min()

### 7.3 Tear Sheet e Visualizzazione dell'Equity Curve
Il blocco finale genera un report riassuntivo delle metriche finanziarie (Win Rate, Maximum Drawdown e Rendimento Netto) e renderizza il grafico dell'Equity Curve. L'andamento della curva permette di valutare visivamente la progressiva erosione del capitale, dimostrando in modo inequivocabile l'impatto dello spread e dei costi di transazione annulli sui margini di profitto in un regime operatività ad altissima frequenza, a meno dell'implementazione di rigidi filtri quantitativi.

In [ ]:
# Calcolo statistiche finali di trading
total_trades = len(df_test)
winning_trades = len(df_test[df_test['Net_PnL_USD'] > 0])
win_rate = (winning_trades / total_trades) * 100
final_equity = df_test['Equity_Curve'].iloc[-1]
net_profit = final_equity - INITIAL_CAPITAL
return_pct = (net_profit / INITIAL_CAPITAL) * 100

print("\n--------------------------------------------------")
print("REPORT FINANZIARIO OUT-OF-SAMPLE (Tear Sheet)")
print("--------------------------------------------------")
print(f"Capitale Iniziale : ${INITIAL_CAPITAL:,.2f}")
print(f"Capitale Finale   : ${final_equity:,.2f}")
print(f"Profitto Netto    : ${net_profit:,.2f} ({return_pct:.2f}%)")
print(f"Drawdown Massimo  : ${max_drawdown:,.2f}")
print(f"Trade Eseguiti    : {total_trades:,}")
print(f"Win Rate (Netto)  : {win_rate:.2f}% (Dopo spread di {SPREAD} punti)")
print("--------------------------------------------------\n")

# Configurazione del grafico dell'Equity Curve
plt.figure(figsize=(12, 6))
plt.plot(df_test['Datetime'], df_test['Equity_Curve'], color='green', linewidth=1.5, label='XGBoost Strategy')
plt.axhline(INITIAL_CAPITAL, color='black', linestyle='--', linewidth=1, label='Break Even')

# Riempimento rosso per i periodi in perdita (sotto il capitale inziale)
plt.fill_between(df_test['Datetime'], INITIAL_CAPITAL, df_test['Equity_Curve'], 
                 where=(df_test['Equity_Curve'] < INITIAL_CAPITAL), color='red', alpha=0.3)

plt.title("Equity Curve: XGBoost Strategy (Out-of-Sample)", fontsize=14, fontweight='bold')
plt.xlabel("Data (Ultimo 20% della serie storica)", fontsize=12)
plt.ylabel("Controvalore del Portafoglio (USD)", fontsize=12)
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()